# Test Graph Loader

End-to-end test of `DataLoaderGraph`: mock data → universal `GraphData`.

In [ ]:
import pandas as pd

from gbp.loaders import DataLoaderMock, DataLoaderGraph, GraphLoaderConfig
from gbp.graph import validate_graph, GraphDataWithQueries

## 1. Load data

In [ ]:
mock = DataLoaderMock({"n": 8, "n_depots": 2, "n_timestamps": 48})

loader = DataLoaderGraph(mock, GraphLoaderConfig(distance_backend="haversine"))
loader.load_data()

## 2. Snapshot

In [ ]:
date = loader.available_dates[12]
graph = loader.get_snapshot(date)
print(graph)

## 3. Inspect static parts

In [ ]:
graph.nodes

In [ ]:
graph.coordinates

In [ ]:
graph.resources

In [ ]:
graph.commodities

In [ ]:
graph.edges.head(10)

## 4. Node attributes

In [ ]:
for name, attr in graph.node_attributes.items():
    print(attr)
    display(attr.data)

## 5. Inventory snapshot

In [ ]:
graph.inventory

## 6. Temporal comparison

In [ ]:
dates = loader.available_dates[[0, 12, 24, 47]]

for d in dates:
    snap = loader.get_snapshot(d)
    total_qty = snap.inventory["quantity"].sum()
    print(f"{d}  total_quantity={total_qty}")

## 7. Inventory time-series (raw)

In [ ]:
loader.inventory_timeseries.head(10)

## 8. Validate graph

In [ ]:
result = validate_graph(graph)
print(result)

## 9. Distance service

In [ ]:
ids = list(graph.nodes["id"][:3])

for src in ids:
    for tgt in ids:
        if src != tgt:
            d = graph.distance_service.get_distance(src, tgt)
            print(f"{src} → {tgt}: {d:.2f} km")